# 05 — Variant 5: Pix2Pix + global discriminator

In [ ]:
import sys
sys.path.append("..")

import json
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from src.data import AnimeColorizationDataset
from src.training import Pix2PixTrainer
from src.utils import seed_everything, plot_loss_curves, qualitative_grid_compare, evaluate_model

seed_everything(42)

DATA_ROOT = Path("../data/anime_colorization")
CKPT_DIR  = Path("../checkpoints/05_pix2pix_global_disc")
FIG_DIR   = Path("../results/figures")
TABLE_DIR = Path("../results/tables")
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Keep in sync with scripts/train_05_pix2pix_global_disc.py
CONFIG = dict(
    image_size=256,
    batch_size=16,
    num_workers=12,
    epochs=100,
    lr=2e-4,
    lambda_l1=100.0,
    lambda_gan=1.0,
    lambda_perceptual=0.0,
    discriminator_type="global",
    early_stopping=False,
)

## Data

In [ ]:
splits = {split: AnimeColorizationDataset(DATA_ROOT, split=split,
                                          image_size=CONFIG["image_size"])
          for split in ("train", "val", "test")}

train_loader = DataLoader(splits["train"], batch_size=CONFIG["batch_size"],
                          shuffle=True,  num_workers=CONFIG["num_workers"],
                          pin_memory=True, drop_last=True, persistent_workers=True)
val_loader   = DataLoader(splits["val"],   batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=CONFIG["num_workers"],
                          pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(splits["test"],  batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=CONFIG["num_workers"],
                          pin_memory=True, persistent_workers=True)

for split, ds in splits.items():
    print(f"{split:5s}: {len(ds):6d} images")

## Training

Training was run headless via `scripts/train_05_pix2pix_global_disc.py`.
This notebook only loads the resulting checkpoints for evaluation.

In [ ]:
def make_trainer():
    return Pix2PixTrainer(
        train_loader=train_loader,
        val_loader=val_loader,
        checkpoint_dir=CKPT_DIR,
        lr=CONFIG["lr"],
        lambda_l1=CONFIG["lambda_l1"],
        lambda_gan=CONFIG["lambda_gan"],
        lambda_perceptual=CONFIG["lambda_perceptual"],
        discriminator_type=CONFIG["discriminator_type"],
        early_stopping=CONFIG["early_stopping"],
    )

trainer = make_trainer()
trainer.load_checkpoint("last.pt")
print(f"last.pt — epoch {trainer.epoch}/{CONFIG['epochs']} | best val_l1={trainer.best_metric:.4f}")

## Loss curves

In [ ]:
plot_loss_curves(trainer.history,
                 title="Variant 5 — Pix2Pix + global discriminator",
                 save_path=FIG_DIR / "05_pix2pix_global_disc_losses.png")

## Qualitative results — last.pt vs best.pt

In [ ]:
batch = next(iter(test_loader))

epoch_last = trainer.epoch
pred_last  = trainer.generate(batch["sketch"])

trainer.load_checkpoint("best.pt")
epoch_best = trainer.epoch
pred_best  = trainer.generate(batch["sketch"])

qualitative_grid_compare(
    batch["sketch"], pred_last, pred_best, batch["color"],
    epoch_last=epoch_last, epoch_best=epoch_best, n_rows=4,
    save_path=FIG_DIR / "05_pix2pix_global_disc_grid.png",
)

## Quantitative evaluation (test set)

In [ ]:
trainer.load_checkpoint("last.pt")
metrics_last = evaluate_model(trainer.generator, test_loader,
                              fid_dir="../results/fid/05_pix2pix_global_disc_last")

trainer.load_checkpoint("best.pt")
metrics_best = evaluate_model(trainer.generator, test_loader,
                              fid_dir="../results/fid/05_pix2pix_global_disc_best")

header = f"{'metric':>8} | {'last.pt':>10} | {'best.pt':>10} | {'Δ (best−last)':>14}"
print(header)
print("-" * len(header))
for k in metrics_last:
    delta = metrics_best[k] - metrics_last[k]
    print(f"{k:>8} | {float(metrics_last[k]):>10.4f} | {float(metrics_best[k]):>10.4f} | {float(delta):>+14.4f}")

with open(TABLE_DIR / "05_pix2pix_global_disc.json", "w") as f:
    json.dump({"variant": "05_pix2pix_global_disc", "config": CONFIG,
               "metrics": {"last": metrics_last, "best": metrics_best}}, f, indent=2)